# AI Inventory Intelligence & Supplier Orchestration 📦🛡️
Run the full Streamlit Dashboard directly from Google Colab!

**Instructions:**
1. Run the first cell to setup the environment.
2. Add your API keys in the second cell.
3. Run the final cell to launch. **Click the 'Universal Dashboard Link' to open the app (Works in all browsers, including Safari).**

In [ ]:
!git clone https://github.com/lmudu2/ai-inventory-intelligence.git
%cd ai-inventory-intelligence
!pip install streamlit pandas plotly langgraph langchain-groq langchain-core groq python-dotenv requests numpy sendgrid

# Install Cloudflare Tunnel (Standard for Universal Browser Compatibility)
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
import os
# ⚠️ ENTER YOUR STAGING API KEYS HERE ⚠️
os.environ['GROQ_API_KEY'] = 'gsk_...'
os.environ['SENDGRID_API_KEY'] = 'SG...'
os.environ['SENDGRID_FROM_EMAIL'] = 'your_email@domain.com'
os.environ['SENDGRID_TO_EMAIL'] = 'target_email@domain.com'
os.environ['NEWSDATA_API_KEY'] = 'pub_...'

print("Environment Variables Set!")

In [ ]:
import time, socket, re
from google.colab import output

def wait_for_port(port, timeout=45):
    start_time = time.time()
    while True:
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=1): return True
        except (OSError, ConnectionRefusedError):
            if time.time() - start_time > timeout: return False
            time.sleep(1)

print("🧹 Cleaning up existing sessions...")
!fuser -k 8501/tcp

print("🚀 Launching Dashboard with Safari-Compatibility Flags...")
get_ipython().system_raw("streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.enableCORS False --server.enableXsrfProtection False --server.enableWebsocketCompression False > streamlit.log 2>&1 &")

if wait_for_port(8501):
    print("✅ Backend Ready! Starting Universal Cloudflare Tunnel...")
    get_ipython().system_raw("cloudflared tunnel --url http://localhost:8501 --no-autoupdate > cloudflare.log 2>&1 &")
    
    # Extract the .trycloudflare.com link
    time.sleep(10)
    with open("cloudflare.log") as f:
        log_content = f.read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
        if match:
            public_url = match.group(0)
            print("\n--- 🎉 UNIVERSAL ACCESS (Works in Safari, Chrome, Firefox) ---")
            print(f"🔗 Universal Dashboard Link: {public_url}")
            print("-----------------------------------------------------------\n")
        else:
            print("⏳ Tunnel still initializing, wait 5s and run '!cat cloudflare.log' to see your link.")
    
    # Native iframe as secondary preview
    from google.colab import output
    output.serve_kernel_port_as_iframe(8501)
else:
    print("❌ Streamlit failed to start. Logs:")
    !cat streamlit.log